# SAM Instance Debugger

Build the cache first with `build_rgb_cache.py`, then use this notebook to step the standalone SAM-instance algorithm frame by frame.

The notebook intentionally keeps the control surface small:
- adjust the hyperparameters in one cell
- recreate the debugger after edits
- use the widget to step / seek / jump to the next seed frame
- inspect detailed decisions and bucket state in the text panel

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [ ]:
from pathlib import Path

from map_runtime.sam_instance_debug import CachedSAMInstanceDebugger, create_debugger_widget
from map_runtime.sam_instance_runtime import SAMInstancePipelineConfig, SAMInstanceRuntimeConfig
from map_runtime.sam_masks import SAMMaskExtractorConfig
from map_runtime.sam2_tracking import SAMTrackerConfig

In [ ]:
# ---------------------------------------------------------------------------------->
# 'metrics': {'instance': {'ap_25': 0.6863656797221245,
#    'ap_50': 0.25332610109808845,
#    'ap_55': 0.16935183297525888,
#    'ap_60': 0.16935183297525888,
#    'ap_65': 0.13552595124960143,
#    'ap_70': 0.09875641005535779,
#    'ap_75': 0.028703456855949525,
#    'ap_80': 0.015053763440860218,
#    'ap_85': 0.00498533724340176,
#    'ap_90': 0.0009775171065493646,
#    'ap_95': 0.0,
#    'ap': 0.08760322030003262}},

# 'metrics': {'instance': {'ap_25': 0.9393939393939394,
#    'ap_50': 0.7121212121212122,
#    'ap_55': 0.7121212121212122,
#    'ap_60': 0.6058802308802309,
#    'ap_65': 0.5123075998075999,
#    'ap_70': 0.42472823472823473,
#    'ap_75': 0.24911856661856657,
#    'ap_80': 0.13594276094276092,
#    'ap_85': 0.11712361712361712,
#    'ap_90': 0.04224386724386724,
#    'ap_95': 0.03593073593073593,
#    'ap': 0.35475180375180376}},
# ---------------------------------------------------------------------------------->

CACHE_DIR = Path("data/output/rgb_caches/ScanNet/scene0011_00")

# seed_mask_config = SAMMaskExtractorConfig(
#     model_level=13,
#     sort_mode="area",
#     min_mask_area_perc=0.01,
#     points_per_side=24,
#     points_per_batch=128,
#     pred_iou_thresh=0.88,
#     stability_score_thresh=0.92,
#     stability_score_offset=1.0,
#     mask_threshold=0.0,
#     box_nms_thresh=0.7,
#     crop_n_layers=0,
#     crop_nms_thresh=0.7,
#     crop_overlap_ratio=0.6,
#     crop_n_points_downscale_factor=1,
#     point_grids=None,
#     min_mask_region_area=0,
#     output_mode="binary_mask",
#     use_m2m=False,
#     multimask_output=True,
#     score_pred_iou_power=2.0,
#     score_stability_power=1.0,
#     score_area_power=0.0,
#     mask_overlap_rescore_thresh=0.0,
#     mask_overlap_rescore_power=1.0,
#     mask_dedupe_iou_thresh=0.85,
#     mask_containment_thresh=0.0,
# )
# tracker_config = SAMTrackerConfig(
#     model_level=24,
#     max_num_objects=16,
# )

# pipeline_config = SAMInstancePipelineConfig(
#     point_gid_slots=10,
#     reuse_inside_frac_th=0.40,
#     reuse_outside_frac_th=0.10,
#     min_mask_points=1,
#     min_track_visible_points=1,
#     prune_start_at=0,  # 400
#     prune_every_frames=64,
#     prune_min_support_perc=0.003,  # 0.004
#     prune_min_points_perc=0.00035,  # 0.0003, roughly 2000 points for 7e6 total points
# )

# config = SAMInstanceRuntimeConfig(
#     seed_mask=seed_mask_config,
#     tracker=tracker_config,
#     pipeline=pipeline_config,
# )

config = SAMInstanceRuntimeConfig()

DEVICE = "cuda"

In [ ]:
debugger = CachedSAMInstanceDebugger(CACHE_DIR, config, device=DEVICE)
create_debugger_widget(debugger)

In [ ]:
debugger.show_sam_masks()

In [ ]:
debugger.show(gid=20)

In [ ]:
# Bucket / point-membership queries on the current debugger state.
# Run debugger.step(), debugger.seek(...), or use the widget first so debugger.current_view is populated.
#
# gid = 3
# bucket = debugger.buckets[gid]
# bucket
#
# point_id = 12345
# debugger.point_gids[point_id]  # K=10 gid slots for one global 3D point
#
# row, col = 200, 300
# point_id = int(debugger.current_view["point_ids_after"][row, col])
# if point_id >= 0:
#     print("pixel -> point_id", point_id)
#     print("gid slots", debugger.point_gids[point_id])
#
# active_buckets = {gid: debugger.buckets[gid] for gid in debugger.current_view["seeded_gids"]}
# active_buckets


In [ ]:
# Manual inspection examples
# debugger.step()
# debugger.seek(80)
# print(debugger.current_text_summary())
# debugger.current_view["decisions"]
# debugger.show(local_id=3)  # seed frames only
# debugger.show(gid=3)       # current projected global-instance support for gid=3
# debugger.show(local_id=3, gid=3)  # compare seed local mask vs current projected gid support
# debugger.simulate_sam_onlyseed_video(upto=100, map_every=1)
# debugger.simulate_video(upto=-1)

In [ ]:
debugger.plots_videos(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000)

In [ ]:
debugger.get_gid_stats(gid=7)

In [ ]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000, use_optimal_text_matching=True, write_label_video=False)

In [ ]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000, use_optimal_text_matching=True, use_collapse_mode="optimal", write_label_video=False)

In [ ]:
debugger.fit(
    scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans",
    min_component_size=2000,
    embed_dim=16,
    num_layers=3,
    epochs=2500,
    pruning_thresh=0.8,
    loss_mode="wbce",  # current best learned frontier
    focal_gamma=2.0,
    plot_every=100,
    show_progress=True,
)


In [ ]:
fit_summary = debugger.debug_last_learned_collapse_summary
gt_selected_gids = fit_summary["target_selected_gids"]
learned_selected_gids = fit_summary["learned_pred_selected_gids"]

In [ ]:
# threedshow usage examples.
# 1) predicted gid only
# debugger.threedshow(gid=5)

# 2) GT instance only
# debugger.threedshow(
#     gt_id=12,
#     scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans",
# )

# 3) predicted gid + GT instance together
# debugger.threedshow(
#     gid=5,
#     gt_id=12,
#     scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans",
# )



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fit_summary = debugger.debug_last_learned_collapse_summary
if fit_summary is None:
    raise RuntimeError("Run debugger.fit(...) first.")

rows = fit_summary["learned_rows"]
gids = np.array([row["gid"] for row in rows], dtype=np.int32)
keep_probs = np.array([row["keep_prob"] for row in rows], dtype=np.float32)
labels = np.array([row["selected"] for row in rows], dtype=bool)
pred_labels = np.array([row["pred_selected"] for row in rows], dtype=bool)
thresh = float(fit_summary["learned_pruning_thresh"])
gt_selected = int(labels.sum())
gt_non_selected = int((~labels).sum())
pred_selected = int(pred_labels.sum())
pred_non_selected = int((~pred_labels).sum())

plt.figure(figsize=(12, 4.5))
plt.scatter(gids[~labels], keep_probs[~labels], s=18, c="red", alpha=0.75, label="non-selected")
plt.scatter(gids[labels], keep_probs[labels], s=18, c="blue", alpha=0.75, label="selected")
plt.axhline(thresh, color="black", linestyle=":", linewidth=2, label=f"thresh={thresh:.3f}")
plt.xlabel("gid")
plt.ylabel("keep probability")
plt.title(
    f"Learned keep probabilities | GT sel/non={gt_selected}/{gt_non_selected} | "
    f"Pred sel/non={pred_selected}/{pred_non_selected} @ thresh={thresh:.3f}"
)
plt.ylim(-0.02, 1.02)
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
debugger.get_metrics(
    scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans",
    min_component_size=2000,
    ovo_score_th=0.0,
    chunk_size=100_000,
    use_optimal_text_matching=True,
    use_collapse_mode="learned",  # requires debugger.fit(...) on the current state first
    write_label_video=False,
)
